# Reinforcement-Learning-Guided Adaptive Temporal–Spectral Feature Fusion for Heart-Sound Analysis

**Satya Krishna V.** · single-dataset study on **CirCor DigiScope v1.0.3** (PhysioNet) ·
integrated project for **24AIM304 Reinforcement Learning** and **24AIM301 Signal & Image Processing**

---

## 1. Problem

Phonocardiogram (PCG) classifiers normally fix their representation by hand: one wavelet, one
frequency band, one analysis window, one fusion rule. Murmurs differ in pitch (low / medium /
high), timing (early, mid, holo-systolic, diastolic) and loudness (grade I–IV/VI), and
recordings differ in noise and auscultation site, so a single hand-picked representation is
unlikely to be best for every recording.

**Question.** Can a reinforcement-learning agent learn, from label-free signal descriptors, *which*
signal representation to compute for each recording — wavelet family, decomposition level,
frequency band, temporal window and fusion of temporal / spectral / wavelet branches — so that an
attention-fusion network detects murmurs better, more robustly and more cheaply than any fixed choice?

**Primary task:** recording-level murmur *Present* vs *Absent* (Unknown excluded and reported
separately). **Secondary:** patient-level aggregation and clinical-outcome ranking.

## 2. The two reference papers and how the project combines them

| Paper | Key idea | Where it lives in this project |
|---|---|---|
| Xiao & Wang, *Adaptive wavelet base selection for deep learning-based ECG diagnosis: a reinforcement learning approach*, PLOS ONE 20(2): e0318070, 2025 | wavelet base chosen per signal by a policy trained with policy gradient; reward = classifier accuracy; baselines CV-WBS and EE-WBS | Phase 9 (Notebook 08): the same problem on PCG as a (contextual) bandit — gradient bandit = tabular policy gradient — compared against CV-WBS and EE-WBS; generalised in Phases 10–12 from *wavelet only* to the full representation |
| *MST-ResFLNet: Multiscale Temporal–Spectral Feature Fusion for Heart Sound Signal Analysis*, IEEE, 2025 | ResNet-18 with a multi-branch pyramid-spectrum stem + SE attention, temporal attention pooling instead of GAP, focal loss for class imbalance | Phases 8 & 11 (Notebooks 14–15): every 2-D branch uses the pyramid stem, SE and temporal attention pooling; loss = weighted BCE + focal loss; the multiscale fusion becomes an **attention fusion of three branches whose inputs RL selects** |

## 3. Course coverage

**24AIM304 Reinforcement Learning**

| Unit / CO | Topic | Notebook |
|---|---|---|
| U1 · CO1 | elements of RL: agent, environment, state, action, reward, return; limitations | 06, 07, 08, 10 |
| U2 · CO2 | multi-armed bandits (ε-greedy, UCB, gradient bandit, Thompson sampling) | 09 |
| U2 · CO2 | finite MDP formulation; DP: policy evaluation, policy improvement, policy iteration, value iteration | 10 |
| U3 · CO3 | Monte Carlo prediction & control (first-visit, ε-soft), incremental implementation | 11 |
| U3 · CO3 | TD prediction, SARSA, Expected SARSA, Q-learning, Double Q-learning, n-step bootstrapping | 11 |
| U4 · CO4 | models and planning, Dyna-Q, prioritized sweeping, trajectory sampling, heuristic search (MCTS/UCT), rollout algorithms | 12 |
| CO5 | RL in a real-world application — representation selection for clinical murmur detection | 14, 16–20 |
| beyond syllabus | PPO with a multi-categorical action (the draft plan's algorithm) as comparison | 13 |

**24AIM301 Signal & Image Processing**

| Unit / CO | Topic | Notebook |
|---|---|---|
| U1 · CO1/CO2 | biomedical signal characteristics, noise, IIR (Butterworth) vs FIR filters, spike removal | 03, 04 |
| U1 · CO1 | power spectral density (Welch), cross-spectral density & coherence | 04 |
| U1 · CO1 | cepstrum & homomorphic filtering (homomorphic envelogram, cepstral heart rate) | 03, 04 |
| U2 · CO1 | Fourier transform (STFT), wavelet transform (db/sym/coif wavelet packets) | 05, 07, 15, 16 |
| U2 · CO3 | PCA (RL state embedding), ICA (source separation) | 04, 08 |
| U3 · CO4 | TF images: sampling/quantisation, histogram equalisation, Otsu segmentation, GLCM texture, pattern classification | 04, 15–16 |
| U4 · CO5 | image compression: redundancy, Shannon entropy & first theorem, fidelity (PSNR), DCT transform coding | 19 |

## 4. Refinements to the draft plan (what was changed, and why)

| Draft plan | Final project | Reason |
|---|---|---|
| "5,272 recordings from 1,568 subjects" | **3,163 recordings from 942 patients** | only the public training portion of CirCor is downloadable; the rest was hidden Challenge data |
| `wget -r -N -c -np …` | same file set mirrored with 24 parallel connections and SHA-256 verification (Notebook 01) | `wget` absent on Windows; PhysioNet throttles each connection |
| patient-level murmur label on every recording | **recording-level label**: Present only if the location is in *Murmur locations* | 19 % of recordings of murmur patients are at sites where no murmur was heard — label noise otherwise |
| windows 0.25 / 0.5 / 1 / 2 s | **0.5 / 1 / 2 / 4 s** | 0.25 s is shorter than one pediatric systole + diastole |
| "selected CWT/DWT" for db / sym / coif | **wavelet packets in frequency order** | a CWT is not defined for these discrete wavelets; packets give a true TF image whose level trades time vs frequency resolution |
| "compact CNN embedding" in the state | **PCA of the 32-band log-PSD** | a supervised embedding leaks labels into the state and behaves differently on seen vs unseen recordings |
| PPO over the full action as the main agent | **finite MDP over the 5 decisions, solved with DP / MC / TD / planning**; PPO kept as comparison | tabular methods are the course content; the sequential decomposition shares value across the 2,100 configurations |
| reward = validation macro-F1 of a retrained classifier | **per-recording, out-of-fold reward** whose expectation equals balanced accuracy / sensitivity / specificity, + noise robustness − complexity, as Δ vs the fixed baseline | macro-F1 is not decomposable per sample; retraining a CNN per query is infeasible |
| (classifier that pays the reward unspecified) | **weight-sharing supernet**: one fusion network trained under random configurations scores all 2,100 configurations; a classical logistic-regression oracle is kept as comparison / ablation | the reward must come from the network family the policy will drive, otherwise the chosen representation need not transfer |
| fine-tuning from the frozen-policy model | fine-tuning **from the supernet** under the frozen policy, with a supernet-fine-tuned *fixed-configuration* control (B6) | separates the effect of the policy from the effect of extra pre-training |
| "fusion strategy" | **branch subset** {T, S, W, TS, TW, SW, TSW} chosen by RL; attention vs concatenation as an ablation | makes fusion an action with a real cost/benefit |
| no statistics | 3 seeds, mean ± std, paired patient-bootstrap CI of the AUROC gain, validation-only model selection | honest comparisons |

## 5. System

```
  CirCor WAV 4 kHz ──► QC ──► trim to TSV span ──► resample 2 kHz ──► Butterworth 20–800 Hz ──► spike removal ──► median/IQR
                                                                                                                    │
  label-free descriptors (SNR, energy, spectral shape, ZCR, cepstral HR, log-PSD) ──► PCA ──► k-means ──► context s = k
                                                                                                                    │
  RL agent (bandit / DP / MC / TD / Dyna / MCTS)  ──►  a = (wavelet, level, band, window, fusion) = π(k)
                                                                                                                    │
      T branch: raw PCG window ──► multi-scale 1-D CNN + SE + temporal attention pooling ──► F_T
      S branch: STFT rows in band ──► pyramid-spectrum stem + SE + ResNet + temporal attention pooling ──► F_S
      W branch: wavelet-packet (wavelet, level) rows in band ──► same 2-D design ──► F_W
                                                                                                                    │
  attention fusion  F = α_T F_T + α_S F_S + α_W F_W  (softmax over the branches enabled by a)  ──► MLP ──► p(murmur)
                                                                                                                    │
  reward  r = .40 Δbal + .25 Δsens + .20 Δspec + .10 robustness − .05 complexity  ──► policy update
```

## 6. RL formulation

| | definition |
|---|---|
| state | discrete context k ∈ {0..K−1} (K chosen on validation) and the prefix of components already chosen |
| actions | wavelet {db4, db6, db8, sym4, coif3} → level {3, 4, 5} → band {20–100, 100–200, 200–400, 400–800, 20–800 Hz} → window {0.5, 1, 2, 4 s} → fusion {T, S, W, TS, TW, SW, TSW} — 2,100 configurations |
| transitions | deterministic prefix extension; stochasticity = which recording is drawn |
| reward | terminal, per recording, out-of-fold; Δ vs baseline (db6, L5, 20–800 Hz, 2 s, TSW) |
| horizon / discount | H = 5, γ = 1 |

## 7. Notebooks (run in order; each one only reads what earlier ones wrote)

| # | notebook | plan phase |
|---|---|---|
| 01 | download & verify CirCor v1.0.3 | 1 |
| 02 | metadata table, recording-level labels, patient-independent split | 2–3 |
| 03 | quality control and preprocessing | 4–6 |
| 04 | signal & image-processing analysis (filters, PSD, cepstrum, coherence, PCA/ICA, TF image processing) | SIP |
| 05 | candidate representations and configuration features | 7 |
| 06 | classical reward oracle (logistic regression) | 12 |
| 07 | deep reward oracle: weight-sharing supernet | 12 |
| 08 | RL state: discrete signal contexts | 10 |
| 09 | bandit wavelet selection (RLWBS on PCG) | 9 |
| 10 | configuration MDP and dynamic programming | 10 |
| 11 | Monte Carlo and TD control | 10 |
| 12 | planning: Dyna-Q, prioritized sweeping, rollout, MCTS | 10 |
| 13 | PPO comparison | 10 |
| 14 | agent selection and RL ablations | 12, 15 |
| 15 | deep baselines | 8 |
| 16 | RL-guided fusion networks and fine-tuning | 11, 13 |
| 17 | deep ablations | 15 |
| 18 | final evaluation on the test split | 14 |
| 19 | robustness: noise and TF-image compression | 14–15 |
| 20 | explainability | 15 |
| 21 | results summary and deliverables | — |

`notebooks/common/` holds the shared library notebooks (configuration, signal processing, RL,
deep learning) that each phase notebook loads with `%run`.

## 8. Dataset and citation

Oliveira J, Renna F, Costa P, *et al.* **The CirCor DigiScope Dataset: From Murmur Detection to
Murmur Classification.** IEEE Journal of Biomedical and Health Informatics 26(6):2524–2535, 2022.
doi:10.1109/JBHI.2021.3137048 · PhysioNet: https://physionet.org/content/circor-heart-sound/1.0.3/
(Goldberger *et al.*, PhysioBank, PhysioToolkit and PhysioNet, Circulation 101(23), 2000).
License ODC-By 1.0. **All experiments use CirCor v1.0.3 only.**